<a href="https://colab.research.google.com/github/anshikaktr/Titanic-Survival-Prediction/blob/main/notebooks/Titanic_Survival_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train_data = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
train_data.head()

In [ ]:
train_data.info()

In [ ]:
train_data.describe()

In [ ]:
train_data['Title'] = train_data['Name'].str.extract(
    r' ([A-Za-z]+)\.',
    expand=False
)

In [ ]:
train_data['Title'].value_counts()

In [ ]:
train_data['Title'] = train_data['Title'].replace({
    'Mlle':'Miss',
    'Ms':'Miss',
    'Mme':'Mrs'
})

In [ ]:
rare_titles = [
    'Lady', 'Countess', 'Capt', 'Col', 'Don',
    'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'
]

In [ ]:
train_data['Title'] = train_data['Title'].replace(rare_titles, 'Rare')

In [ ]:
train_data['Title'] = train_data['Title'].map({
    'Mr':1,
    'Miss':2,
    'Mrs':3,
    'Master':4,
    'Rare':5
})

In [ ]:
train_data.head()

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data["Age"]=train_data["Age"].fillna(train_data["Age"].median())

In [ ]:
train_data["Embarked"]=train_data["Embarked"].fillna(train_data["Embarked"].mode()[0])

In [ ]:
train_data.drop(columns=["Cabin"], inplace=True)

Missing values in Age were replaced using the median age.
Missing values in Embarked were replaced using the most frequent embarkation port.
Cabin was removed because a large proportion of values were missing.

In [ ]:
train_data.isnull().sum()

In [ ]:
train_data["Survived"].mean()

In [ ]:
train_data["Survived"].mean() *100

#### Finding 1: Overall Survival Rate

Out of all passengers in the dataset, approximately 38% survived the Titanic disaster, while about 62% did not survive.

In [ ]:
train_data.groupby("Sex")["Survived"].mean()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(data=train_data, x="Sex", y="Survived")
plt.title("Survival rate by Gender")
plt.show()

#### Finding 2: Survival by Gender

Female passengers had a much higher survival rate (74%) than male passengers (19%).

This suggests that gender was one of the strongest factors influencing survival during the Titanic disaster.

In [ ]:
train_data.groupby("Pclass")["Survived"].mean()

In [ ]:
sns.barplot(data=train_data, x="Pclass", y="Survived")
plt.title("Survival rate by Ticket Class")
plt.show()

#### Finding 3: Survival by Passenger Class

Ticket class had a strong relationship with survival.

First-class passengers had the highest survival rate (63%), followed by second-class passengers (47%). Third-class passengers had the lowest survival rate (24%).

This suggests that socioeconomic status may have influenced access to lifeboats and evacuation opportunities.

In [ ]:
train_data.groupby("Age")["Survived"].mean()

In [ ]:
train_data.groupby("Survived")["Age"].mean()

In [ ]:
sns.boxplot(data=train_data, x="Survived", y="Age")
plt.title("Age Distribution by Survival Status")
plt.show()

#### Finding 4: Age and Survival

The average age of survivors (28.3 years) was slightly lower than that of non-survivors (30.0 years).

However, the difference was relatively small, suggesting that age alone was not as strong a predictor of survival as gender or ticket class.

In [ ]:
train_data.groupby("Survived")["Fare"].mean()

In [ ]:
sns.boxplot(data=train_data, x="Survived", y="Fare")
plt.title("Fare Distribution by Survival Status")
plt.show()

#### Finding 5: Fare and Survival

Passengers who survived generally paid higher fares than those who did not survive.

This suggests that wealth and passenger class may have played an important role in survival outcomes.

In [ ]:
train_data.groupby("Embarked")["Survived"].mean()

In [ ]:
sns.barplot(data=train_data, x="Embarked", y="Survived")
plt.title("Survival rate by Embarkation Port")
plt.show()

#### Finding 6: Embarkation Port and Survival

Passengers who embarked from Cherbourg (C) had the highest survival rate at approximately 55%, compared to 39% for Queenstown (Q) and 34% for Southampton (S).

This may be related to passenger class distribution, as wealthier passengers were more likely to board from certain ports.

In [ ]:
train_data["FamilySize"] = train_data["SibSp"] + train_data["Parch"] + 1

In [ ]:
train_data['IsAlone'] = (
    train_data['FamilySize'] == 1
).astype(int)

In [ ]:
train_data.groupby("FamilySize")["Survived"].mean()

In [ ]:
sns.barplot(data=train_data, x="FamilySize", y="Survived")
plt.show()

In [ ]:
train_data["FamilyType"]=pd.cut(train_data["FamilySize"], bins=[0,1,4,15],
                                labels=["Alone", "Small Family", "Large Family"])

In [ ]:
train_data.groupby("FamilyType", observed=False)["Survived"].mean()

#### Finding 7: Family Size and Survival

Family size appeared to influence survival outcomes.

Passengers travelling in small family groups had the highest survival rate (57.9%), compared to passengers travelling alone (30.4%) and those travelling in large families (16.1%).

This suggests that travelling with a small family may have provided advantages during evacuation, while larger family groups may have faced greater challenges.

In [ ]:
train_model= train_data.copy()

In [ ]:
train_model["Sex"]=train_model["Sex"].map({
    "male":0,
    "female":1
})

train_model["Embarked"]=train_model["Embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})

In [ ]:
train_model = pd.get_dummies(train_model,
               columns=['Embarked'],
               drop_first=True,
              dtype=int)

In [ ]:
train_model.head()

In [ ]:
corr=train_model.corr(numeric_only=True)
corr["Survived"].sort_values(ascending=False)


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Heatmap")
plt.show()

#### Correlation Analysis

Sex showed the strongest positive correlation with survival (0.54), indicating that gender was one of the most important factors associated with survival.

Passenger class showed a moderate negative correlation (-0.34), suggesting that passengers in lower classes were less likely to survive.

Fare showed a moderate positive correlation (0.26), while age exhibited only a weak relationship with survival (-0.06).

The correlation analysis largely supported the patterns observed during exploratory data analysis.

In [ ]:
features = [
    'Pclass',
    'Sex',
    'Age',
    'Fare',
    'Embarked_Q',
    'Embarked_S',
    'FamilySize',
    'Title'
]

In [ ]:
train_model.head()

In [ ]:
X=train_model[features]
y=train_model["Survived"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)

from sklearn.ensemble import RandomForestClassifier
model= RandomForestClassifier(
    n_estimators=100, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

In [ ]:
X_train.describe()

In [ ]:
X_train.head()

In [ ]:
model.fit(X_train,y_train)

In [ ]:
predicts=model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
print("Accuracy:", accuracy_score(y_test, predicts))
print("Confusion Matrix \n", confusion_matrix(y_test, predicts))

In [ ]:
feature_imp = pd.DataFrame({"Feature": features,"Importance": model.feature_importances_})
feature_imp.sort_values(by="Importance",ascending=False)

In [ ]:
model.fit(X,y)

In [ ]:
test_data = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
test_data.head()

In [ ]:
test_data.isnull().sum()

In [ ]:
test_data["Age"]=test_data["Age"].fillna(test_data["Age"].median())

In [ ]:
test_data["Fare"]=test_data["Fare"].fillna(test_data["Fare"].median())

In [ ]:
test_data["Sex"]=test_data["Sex"].map({
    "male":0,
    "female":1
})

In [ ]:
test_data["FamilySize"] = test_data["SibSp"] + test_data["Parch"] + 1

In [ ]:
test_data = pd.get_dummies(test_data,
               columns=['Embarked'],
               drop_first=True,
              dtype=int)

In [ ]:
test_data['Title'] = test_data['Name'].str.extract(
    r' ([A-Za-z]+)\.',
    expand=False
)

In [ ]:
test_data['Title'] = test_data['Title'].replace({
    'Mlle':'Miss',
    'Ms':'Miss',
    'Mme':'Mrs'
})

In [ ]:
rare_titles = [
    'Lady', 'Countess', 'Capt', 'Col', 'Don',
    'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'
]

In [ ]:
test_data['Title'] = test_data['Title'].replace(rare_titles, 'Rare')

In [ ]:
test_data['Title'] = test_data['Title'].map({
    'Mr':1,
    'Miss':2,
    'Mrs':3,
    'Master':4,
    'Rare':5
})

In [ ]:
test_data['IsAlone'] = (
    test_data['FamilySize'] == 1
).astype(int)

In [ ]:
test_features = [
    'Pclass',
    'Sex',
    'Age',
    'Fare',
    'Embarked_Q',
    'Embarked_S',
    'FamilySize',
    'Title'
]

In [ ]:
test_model=test_data[features]

In [ ]:
print(X.columns.tolist())
print(test_model.columns.tolist())

In [ ]:
test_predicts = model.predict(test_model)

In [ ]:
submission6 = pd.DataFrame({
    "PassengerId":test_data["PassengerId"], "Survived":test_predicts
})

In [ ]:
submission6.head()

In [ ]:
submission6.to_csv("submission6.csv",index=False)

In [ ]:
submission6.head()